# Bin Buddy · WS1 · Notebook 02 — Comparative analysis of vision models

**Question:** which model should Bin Buddy use to recognise an item's **material** from a phone
photo, given ~10k public images, ~100–200 local Singapore photos, a free CPU-only deployment,
and a requirement that every verdict comes with a Grad-CAM explanation?

**Design of the comparison (controlled experiment):** every model gets the *same* data, splits,
input size (224 px), augmentation, schedule and metrics (`ml/binbuddy_ml/experiments.py`).
Only the model changes, so differences in the results are caused by the model.

**Experiments**
- **E1 · public only** → tested on the public test split *and* the local test split. The gap
  between the two numbers measures the domain shift to Singapore household photos.
- **E2 · public + local** → the local training photos are added, model selection uses the local
  validation split, tested on the local test split. This is the realistic deployment setting.

**Selection rule — written before seeing any result** (so the choice cannot be cherry-picked):
1. Highest **local-test macro-F1** in E2.
2. If 95% confidence intervals overlap, prefer the lower **costly-error rate** (a non-recyclable
   material predicted as a recyclable one — the error that contaminates a blue bin).
3. Then prefer lower **CPU latency** (free, local deployment), then a model that supports
   Grad-CAM directly (a CNN).

## 0. Setup

Run on **Google Colab** (Runtime → Change runtime type → **T4 GPU** for notebooks 02–04; CPU is
enough for 01). The cell below mounts Google Drive, clones the team repository (branch
`ws1/vision-model`) and installs pinned packages, so every run uses the same code and versions.

`SMOKE=1` is only for the developer's local test run on tiny synthetic data; leave it unset on Colab.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

SMOKE = os.environ.get("BINBUDDY_SMOKE") == "1"  # local test on synthetic data only

# Experiment tracking: every run keeps its own files, so nothing is ever overwritten.
#   Data v1 = public data + 207 team photos (14 Oct)        -> Run 1
#   Data v2 = v1 + the final team photo batch (frozen)      -> Run 2 and later runs
DATA_VERSION = os.environ.get("BINBUDDY_DATA_VERSION", "v2")
RUN_ID = os.environ.get("BINBUDDY_RUN_ID", "run2")
REPO_URL = "https://github.com/DevelopingGod/Group-5-PRS-Practice-Module-2026.git"
BRANCH = "ws1/vision-model"
PINNED = ["timm==1.0.30", "grad-cam==1.5.7", "open-clip-torch==3.3.0", "imagehash==4.3.2"]

if SMOKE:
    REPO = Path(os.environ["BINBUDDY_REPO"])
    DATA = Path(os.environ["BINBUDDY_DATA_ROOT"])
    WORK = DATA / "_work"
else:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path("/content/repo")
    DATA = Path("/content/drive/MyDrive/BinBuddy_Data")
    WORK = Path("/content/work")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=True)

# The contracts and helpers come straight from the repository (no copy-paste drift).
for p in (REPO / "contracts", REPO / "ml", REPO / "services" / "inference"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
WORK.mkdir(parents=True, exist_ok=True)
PREP = WORK / f"prepared_{DATA_VERSION}"  # resized images, extracted from TAR
TAR = DATA / f"prepared_{DATA_VERSION}.tar"
# Run 1 / data v1 used the original, unversioned names; later versions get a suffix.
MANIFEST = (
    DATA
    / "manifests"
    / ("vision_manifest.csv" if DATA_VERSION == "v1" else f"vision_manifest_{DATA_VERSION}.csv")
)
RESULTS = DATA / "results" if RUN_ID == "run1" else DATA / "results" / RUN_ID
MODELS_DIR = DATA / "models" if RUN_ID == "run1" else DATA / "models" / RUN_ID
for d in (MANIFEST.parent, RESULTS, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)
COMMIT = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True
).stdout.strip()
print(f"run {RUN_ID} | data {DATA_VERSION} | code {COMMIT} | smoke {SMOKE}")
print("results ->", RESULTS)

In [ ]:
import tarfile

import pandas as pd

if not PREP.exists():
    with tarfile.open(TAR) as tf:  # one big file from Drive is much faster than 10k small ones
        tf.extractall(PREP, filter="data")
df = pd.read_csv(MANIFEST)
print(len(df), "images")
print(df.groupby(["source", "split"]).size().unstack(fill_value=0))

In [ ]:
import copy
import itertools
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn.functional as F
from PIL import Image
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

from binbuddy_contracts.enums import Material
from binbuddy_ml import experiments as ex

CLASSES = [m.value for m in Material]
RECYCLABLE = {"paper", "plastic", "glass", "metal", "beverage_carton"}  # HDB blue-bin materials
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PRETRAINED = os.environ.get("BINBUDDY_SMOKE_PRETRAINED", "1") == "1"
P = ex.PROTOCOL
if SMOKE:
    P = ex.Protocol(batch_size=8, head_epochs=1, full_epochs=1, mixed_precision=False)
MAX_BATCHES = 2 if SMOKE else None  # SMOKE only: stop each epoch early
torch.manual_seed(P.seed)
np.random.seed(P.seed)
print("device:", DEVICE, "| pretrained weights:", PRETRAINED)
df["label"] = df["material"].map(CLASSES.index)

## 1. The candidates — what each model does and why it is here

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.DataFrame(
    [
        {
            "model": m.key,
            "family": m.family,
            "mode": m.mode.value,
            "params (M)": m.params_m,
            "what it does": m.what,
            "why it is in the comparison": m.why,
        }
        for m in ex.MODELS
    ]
)

**Three ways of using a pretrained network** (all start from weights learned on millions of
images — *transfer learning* — because 10k images are far too few to train from scratch):
- **Fine-tuning** (6 models): replace the 1000-class ImageNet head with a 10-class head, train
  the head alone first (stage 1, so random head gradients do not wreck the pretrained
  features), then train the whole network with a small learning rate (stage 2).
- **Linear probe / "vectorisation"** (DINOv2, CLIP): keep the network frozen, turn every image
  into an embedding vector, and fit a logistic-regression classifier on the vectors. Cheap,
  cannot overfit the backbone to 100 local photos, and the vectors are reusable (verification,
  out-of-distribution checks).
- **Zero-shot** (CLIP): no training images at all; the class is the text prompt whose embedding
  is closest to the image embedding.

## 2. Shared training protocol (hyperparameters and why)

| Hyperparameter | Value | Why |
|---|---|---|
| Input size | 224 × 224 | Native resolution of every pretrained model here |
| Optimiser | AdamW, weight decay 0.05 | Decoupled weight decay regularises well for both CNNs and ViTs |
| Stage 1 (head only) | 2 epochs, lr 1e-3 | Warm up the new head without disturbing pretrained features |
| Stage 2 (full) | ≤ 12 epochs, per-model backbone lr (CNN 1e-4…3e-4, ViT 5e-5), head lr = 10× backbone | Small backbone steps keep ImageNet knowledge; transformers are more sensitive |
| Schedule | 10% linear warm-up, then cosine decay to 0 | Stable start, smooth convergence |
| Loss | Cross-entropy, label smoothing 0.1, class weights (1/freq)^0.5 | Smoothing reduces over-confidence (better calibration); square-root weights help rare classes without over-boosting them |
| Early stopping | patience 3 on validation macro-F1 | Macro-F1 treats every material equally; stops overfitting |
| Augmentation | random resized crop (scale 0.6–1), horizontal flip, rotation ±15°, colour jitter 0.3 | Simulates framing, hand-held angles and indoor lighting in phone photos; no greyscale, because colour carries material information |
| Mixed precision | on (GPU) | ~2× faster on a T4 with no accuracy loss |
| Seed | 42 | Repeatability |

In [ ]:
pd.DataFrame(
    [
        {"model": m.key, "mode": m.mode.value, "backbone lr": m.backbone_lr, "batch": m.batch_size}
        for m in ex.MODELS
    ]
).merge(pd.DataFrame([vars(P)]), how="cross")

In [ ]:
class ImageTable(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(PREP / self.paths[i]) as img:
            return self.transform(img.convert("RGB")), self.labels[i]


def transforms_for(model):
    cfg = timm.data.resolve_data_config({}, model=model)
    norm = T.Normalize(cfg["mean"], cfg["std"])
    s = P.input_size
    train = T.Compose(
        [
            T.RandomResizedCrop(s, scale=(0.6, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomRotation(15),
            T.ColorJitter(0.3, 0.3, 0.3, 0.05),
            T.ToTensor(),
            norm,
        ]
    )
    test = T.Compose([T.Resize(int(s / 0.875)), T.CenterCrop(s), T.ToTensor(), norm])
    return train, test


def loader(frame, transform, shuffle, batch_size):
    workers = 0 if SMOKE else 2
    return DataLoader(
        ImageTable(frame, transform),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=DEVICE == "cuda",
    )


def class_weights(frame):
    freq = np.bincount(frame["label"], minlength=len(CLASSES)).astype(float)
    w = np.where(freq > 0, (1.0 / np.maximum(freq, 1)) ** P.class_weight_power, 0.0)
    return torch.tensor(w / w[freq > 0].mean(), dtype=torch.float32)

## 3. Metrics
- **Macro-F1** (primary): F1 per material, averaged with equal weight, so rare but important
  classes (e-waste, styrofoam) count as much as plastic. Computed over the classes present in
  the test set. **95% CI** by bootstrap (1,000 resamples of the test images).
- **Accuracy** and **top-2 accuracy** (is the right material at least the second guess?).
- **Costly-error rate**: share of truly non-recyclable items (styrofoam, e-waste, textile,
  food, other) predicted as a blue-bin material — the mistake that contaminates a whole bin.
- **ECE** (expected calibration error, 15 bins): does "90% sure" mean right 90% of the time?
  Bin Buddy's confidence gate depends on this.
- **CPU latency** (batch of 1, median of 20) and **parameters**: cost of a free local deployment.

In [ ]:
def ece_score(probs, y, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in itertools.pairwise(edges):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(total)


def metrics(probs, y, n_boot=1000):
    pred = probs.argmax(1)
    labels = sorted(set(y.tolist()))
    f1 = lambda a, b: f1_score(a, b, labels=labels, average="macro", zero_division=0)  # noqa: E731
    rng = np.random.default_rng(0)
    boots = [f1(y[i], pred[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(n_boot))]
    nonrec = np.array([CLASSES[t] not in RECYCLABLE for t in y])
    pred_rec = np.array([CLASSES[p] in RECYCLABLE for p in pred])
    top2 = (np.argsort(-probs, 1)[:, :2] == y[:, None]).any(1)
    return {
        "n": len(y),
        "macro_f1": float(f1(y, pred)),
        "ci_low": float(np.percentile(boots, 2.5)),
        "ci_high": float(np.percentile(boots, 97.5)),
        "accuracy": float((pred == y).mean()),
        "top2_accuracy": float(top2.mean()),
        "costly_error": float(pred_rec[nonrec].mean()) if nonrec.any() else float("nan"),
        "ece": ece_score(probs, y),
        "per_class_f1": dict(
            zip(
                [CLASSES[i] for i in labels],
                f1_score(y, pred, labels=labels, average=None, zero_division=0).tolist(),
                strict=True,
            )
        ),
    }


def cpu_latency_ms(fn, reps=20):
    x = torch.randn(1, 3, P.input_size, P.input_size)
    with torch.no_grad():
        for _ in range(3):
            fn(x)
        times = []
        for _ in range(reps):
            t0 = time.perf_counter()
            fn(x)
            times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))

## 4. Training and evaluation code

In [ ]:
@torch.no_grad()
def predict(model, dl):
    model.eval()
    out = []
    for b, (x, _) in enumerate(dl):
        if MAX_BATCHES and b >= MAX_BATCHES:
            break
        with torch.autocast(DEVICE, enabled=P.mixed_precision and DEVICE == "cuda"):
            out.append(model(x.to(DEVICE)).float().softmax(1).cpu())
    return torch.cat(out).numpy()


def run_epochs(model, params, epochs, train_dl, val_dl, y_val, loss_fn, history, stage):
    opt = torch.optim.AdamW(params, weight_decay=P.weight_decay)
    steps = epochs * (min(len(train_dl), MAX_BATCHES) if MAX_BATCHES else len(train_dl))
    warm = max(1, int(P.warmup_fraction * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (
            (s + 1) / warm
            if s < warm
            else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm)))
        ),
    )
    scaler = torch.amp.GradScaler(enabled=P.mixed_precision and DEVICE == "cuda")
    best_f1, best_state, bad = -1.0, None, 0
    for epoch in range(epochs):
        model.train()
        for b, (x, y) in enumerate(train_dl):
            if MAX_BATCHES and b >= MAX_BATCHES:
                break
            with torch.autocast(DEVICE, enabled=P.mixed_precision and DEVICE == "cuda"):
                loss = loss_fn(model(x.to(DEVICE)), y.to(DEVICE))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
        probs = predict(model, val_dl)
        yv = y_val[: len(probs)]
        val_f1 = f1_score(
            yv, probs.argmax(1), labels=sorted(set(yv)), average="macro", zero_division=0
        )
        history.append(
            {"stage": stage, "epoch": epoch, "loss": float(loss), "val_macro_f1": float(val_f1)}
        )
        print(f"  {stage} epoch {epoch}: loss {float(loss):.3f}  val macro-F1 {val_f1:.3f}")
        if val_f1 > best_f1:
            best_f1, best_state, bad = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= P.early_stop_patience:
                break
    model.load_state_dict(best_state)
    return best_f1


def fine_tune(spec, train_df, val_df):
    model = timm.create_model(
        spec.timm_id, pretrained=PRETRAINED, num_classes=len(CLASSES), **dict(spec.timm_kwargs)
    ).to(DEVICE)
    tr_tf, te_tf = transforms_for(model)
    train_dl = loader(train_df, tr_tf, True, spec.batch_size)
    val_dl = loader(val_df, te_tf, False, spec.batch_size)
    y_val = val_df["label"].to_numpy()
    loss_fn = torch.nn.CrossEntropyLoss(
        weight=class_weights(train_df).to(DEVICE), label_smoothing=P.label_smoothing
    )
    head = list(model.get_classifier().parameters())
    head_ids = {id(p) for p in head}
    body = [p for p in model.parameters() if id(p) not in head_ids]
    history = []
    for p in body:  # stage 1: only the new head learns
        p.requires_grad = False
    run_epochs(
        model,
        [{"params": head, "lr": P.head_lr}],
        P.head_epochs,
        train_dl,
        val_dl,
        y_val,
        loss_fn,
        history,
        "head",
    )
    for p in body:  # stage 2: everything learns, backbone more slowly
        p.requires_grad = True
    groups = [
        {"params": body, "lr": spec.backbone_lr},
        {"params": head, "lr": spec.backbone_lr * 10},
    ]
    best = run_epochs(
        model, groups, P.full_epochs, train_dl, val_dl, y_val, loss_fn, history, "full"
    )
    model.eval()
    return model, te_tf, history, best

In [ ]:
@torch.no_grad()
def embed(model, frame, transform):
    dl = loader(frame, transform, False, 64)
    model.eval()
    feats = []
    for b, (x, _) in enumerate(dl):
        if MAX_BATCHES and b >= MAX_BATCHES:
            break
        feats.append(F.normalize(model(x.to(DEVICE)).float(), dim=1).cpu())
    return torch.cat(feats).numpy()


def linear_probe(spec, train_df, val_df):
    model = (
        timm.create_model(
            spec.timm_id, pretrained=PRETRAINED, num_classes=0, **dict(spec.timm_kwargs)
        )
        .to(DEVICE)
        .eval()
    )
    _, te_tf = transforms_for(model)
    xtr, xva = embed(model, train_df, te_tf), embed(model, val_df, te_tf)
    ytr, yva = train_df["label"].to_numpy()[: len(xtr)], val_df["label"].to_numpy()[: len(xva)]
    best = (-1.0, None, None)
    for c in (0.1, 1.0, 10.0):  # inverse regularisation strength, chosen on validation data
        clf = LogisticRegression(C=c, max_iter=3000, class_weight="balanced").fit(xtr, ytr)
        f1 = f1_score(
            yva, clf.predict(xva), labels=sorted(set(yva)), average="macro", zero_division=0
        )
        if f1 > best[0]:
            best = (f1, c, clf)
    f1, c, clf = best
    print(f"  probe: best C={c}, val macro-F1 {f1:.3f}")

    def predict_probs(frame):
        x = embed(model, frame, te_tf)
        p = np.zeros((len(x), len(CLASSES)))
        p[:, clf.classes_] = clf.predict_proba(x)
        return p

    return model, predict_probs, [{"stage": "probe", "C": c, "val_macro_f1": float(f1)}], f1


def zero_shot():
    import open_clip

    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-16", pretrained="openai" if PRETRAINED else None
    )
    model = model.to(DEVICE).eval()
    tok = open_clip.get_tokenizer("ViT-B-16")
    with torch.no_grad():
        text = []
        for c in CLASSES:
            prompts = [t.format(p) for p in ex.ZERO_SHOT_PROMPTS[c] for t in ex.PROMPT_TEMPLATES]
            e = F.normalize(model.encode_text(tok(prompts).to(DEVICE)).float(), dim=1).mean(0)
            text.append(F.normalize(e, dim=0))
        text = torch.stack(text)

    def predict_probs(frame):
        dl = loader(frame, preprocess, False, 64)
        out = []
        with torch.no_grad():
            for b, (x, _) in enumerate(dl):
                if MAX_BATCHES and b >= MAX_BATCHES:
                    break
                img = F.normalize(model.encode_image(x.to(DEVICE)).float(), dim=1)
                out.append((100.0 * img @ text.T).softmax(1).cpu())
        return torch.cat(out).numpy()

    return model, predict_probs

## 5. Run the experiments
Runs every model in E1 and E2 and saves results, per-image predictions (handed to WS2 for the
ensemble) and the E2 fine-tuned weights to Drive. Re-running skips finished runs, so a Colab
disconnect only loses the model in progress. Expect roughly 1.5–2.5 hours on a T4.

In [ ]:
split = {s: df[df["split"] == s] for s in ("train", "val", "test")}
pub = {s: f[f["source"] != "local"] for s, f in split.items()}
loc = {s: f[f["source"] == "local"] for s, f in split.items()}
HAVE_LOCAL = len(loc["train"]) > 0 and len(loc["test"]) > 0
EXPERIMENTS = {
    "E1_public_only": (
        pub["train"],
        pub["val"],
        {"public_test": pub["test"], "local_test": loc["test"]},
    ),
}
if HAVE_LOCAL:
    EXPERIMENTS["E2_public_plus_local"] = (
        pd.concat([pub["train"], loc["train"]]),
        loc["val"] if len(loc["val"]) else pub["val"],
        # public_test also scores the materials the team photos lack (textile, food, styrofoam)
        {"local_test": loc["test"], "public_test": pub["test"]},
    )
else:
    print("No local photos in the manifest yet: only E1 runs.")
RUN_MODELS = [m.key for m in ex.MODELS]  # edit to re-run a subset
RESULTS_FILE = RESULTS / "comparison_results.json"
# Resume: runs already in the results file are skipped, so a Colab disconnect only loses the
# model that was training at the time. (SMOKE runs start fresh unless BINBUDDY_SMOKE_RESUME=1.)
RESUME = not SMOKE or os.getenv("BINBUDDY_SMOKE_RESUME") == "1"
results = json.loads(RESULTS_FILE.read_text()) if RESUME and RESULTS_FILE.exists() else []
done = {(r["experiment"], r["model"]) for r in results}

In [ ]:
def evaluate(name, spec, predict_probs, tests, latency, extra):
    for test_name, frame in tests.items():
        if not len(frame):
            continue
        probs = predict_probs(frame)
        y = frame["label"].to_numpy()[: len(probs)]
        r = {
            "experiment": name,
            "model": spec.key,
            "family": spec.family,
            "mode": spec.mode.value,
            "params_m": spec.params_m,
            "test_set": test_name,
            "cpu_latency_ms": latency,
            **extra,
            **metrics(probs, y),
        }
        results.append(r)
        pred = pd.DataFrame(probs, columns=CLASSES)
        pred.insert(0, "path", frame["path"].to_numpy()[: len(probs)])
        pred.insert(1, "true", [CLASSES[i] for i in y])
        pred.to_csv(RESULTS / f"pred_{name}_{spec.key}_{test_name}.csv", index=False)
        print(
            f"  {test_name}: macro-F1 {r['macro_f1']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]"
            f"  acc {r['accuracy']:.3f}  costly {r['costly_error']:.3f}  ECE {r['ece']:.3f}"
        )


for name, (train_df, val_df, tests) in EXPERIMENTS.items():
    for key in RUN_MODELS:
        spec = ex.model(key)
        if (name, key) in done or (spec.mode is ex.Mode.ZERO_SHOT and not name.startswith("E1")):
            continue  # zero-shot does not train, so E2 would repeat E1
        print(f"== {name} · {key}")
        t0 = time.time()
        if spec.mode is ex.Mode.FINE_TUNE:
            model, te_tf, history, best = fine_tune(spec, train_df, val_df)
            predict_probs = lambda f, m=model, t=te_tf: predict(m, loader(f, t, False, 64))  # noqa: E731
            if name.startswith("E2"):
                torch.save(model.state_dict(), MODELS_DIR / f"{name}_{key}.pt")
            cpu_model = copy.deepcopy(model).float().cpu().eval()
            latency = cpu_latency_ms(cpu_model)
        elif spec.mode is ex.Mode.LINEAR_PROBE:
            model, predict_probs, history, best = linear_probe(spec, train_df, val_df)
            latency = cpu_latency_ms(copy.deepcopy(model).float().cpu().eval())
        else:
            model, predict_probs = zero_shot()
            history, best = [], float("nan")
            enc = copy.deepcopy(model).float().cpu().eval()
            latency = cpu_latency_ms(enc.encode_image)
        extra = {
            "train_time_s": round(time.time() - t0, 1),
            "best_val_macro_f1": best,
            "history": history,
        }
        evaluate(name, spec, predict_probs, tests, latency, extra)
        if RESUME:
            RESULTS_FILE.write_text(json.dumps(results, indent=1))  # saved after every model
        del model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

## 6. Results
### 6.1 Summary table (sorted by the primary metric)

In [ ]:
res = pd.DataFrame(results)
cols = [
    "experiment",
    "test_set",
    "model",
    "family",
    "macro_f1",
    "ci_low",
    "ci_high",
    "accuracy",
    "top2_accuracy",
    "costly_error",
    "ece",
    "cpu_latency_ms",
    "params_m",
    "train_time_s",
    "n",
]
table = res[cols].sort_values(["experiment", "test_set", "macro_f1"], ascending=[True, True, False])
table.round(3)

### 6.2 Domain gap (E1): public test vs local test
A large drop from public to local test means the model learned dataset-specific cues
(backgrounds, lighting) rather than material — the RealWaste finding, now on Singapore photos.

In [ ]:
e1 = res[res["experiment"] == "E1_public_only"].pivot(
    index="model", columns="test_set", values="macro_f1"
)
if {"public_test", "local_test"} <= set(e1.columns):
    e1["drop"] = e1["public_test"] - e1["local_test"]
    e1.sort_values("drop").plot.barh(
        y=["public_test", "local_test"], figsize=(7, 4), title="E1 macro-F1: public vs local test"
    )
    plt.tight_layout()
    plt.show()
e1.round(3)

### 6.3 Main comparison with 95% confidence intervals

In [ ]:
main_exp = "E2_public_plus_local" if HAVE_LOCAL else "E1_public_only"
main_test = "local_test" if HAVE_LOCAL else "public_test"
main = res[
    (res["experiment"] == main_exp)
    | ((res["mode"] == "zero_shot") & (res["test_set"] == main_test))
]
main = main[main["test_set"] == main_test].sort_values("macro_f1")
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(
    main["model"],
    main["macro_f1"],
    xerr=[main["macro_f1"] - main["ci_low"], main["ci_high"] - main["macro_f1"]],
    capsize=3,
)
ax.set_xlabel(f"macro-F1 on {main_test} ({main_exp})")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(main["cpu_latency_ms"], main["macro_f1"])
for _, r in main.iterrows():
    ax.annotate(r["model"], (r["cpu_latency_ms"], r["macro_f1"]), fontsize=7)
ax.set_xscale("log")
ax.set_xlabel("CPU latency per photo (ms, log scale)")
ax.set_ylabel("macro-F1")
ax.set_title("Accuracy vs cost (upper-left is better)")
plt.tight_layout()
plt.show()

### 6.4 Per-material F1 (which classes are hard, and for whom?)

In [ ]:
pc = pd.DataFrame({r["model"]: r["per_class_f1"] for _, r in main.iterrows()}).T
fig, ax = plt.subplots(figsize=(9, 0.45 * len(pc) + 1))
im = ax.imshow(pc.to_numpy(dtype=float), vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(pc.columns)), pc.columns, rotation=45, ha="right")
ax.set_yticks(range(len(pc)), pc.index)
for (i, j), v in np.ndenumerate(pc.to_numpy(dtype=float)):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6, color="w")
fig.colorbar(im)
plt.tight_layout()
plt.show()

### 6.5 Is the winner *significantly* better than the runner-up? (McNemar's test)
On the same test images, count where exactly one of the two models is right. If both were equally
good, those disagreements would split 50/50; an exact binomial test gives the p-value.

In [ ]:
ranked = main.sort_values("macro_f1", ascending=False)
if len(ranked) >= 2:
    a, b = ranked.iloc[0], ranked.iloc[1]

    def correct(r):
        p = pd.read_csv(RESULTS / f"pred_{r['experiment']}_{r['model']}_{main_test}.csv")
        return p[CLASSES].to_numpy().argmax(1) == p["true"].map(CLASSES.index).to_numpy()

    ca, cb = correct(a), correct(b)
    n01, n10 = int((~ca & cb).sum()), int((ca & ~cb).sum())
    p = binomtest(n10, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    print(
        f"{a['model']} vs {b['model']}: only {a['model']} right {n10}, only {b['model']} right {n01}, p = {p:.3f}"
    )

### 6.6 Confusion matrix and calibration of the top model

In [ ]:
best = ranked.iloc[0]
pred = pd.read_csv(RESULTS / f"pred_{best['experiment']}_{best['model']}_{main_test}.csv")
probs = pred[CLASSES].to_numpy()
y = pred["true"].map(CLASSES.index).to_numpy()
cm = confusion_matrix(y, probs.argmax(1), labels=range(len(CLASSES)))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks(range(len(CLASSES)), CLASSES, rotation=60, ha="right", fontsize=7)
axes[0].set_yticks(range(len(CLASSES)), CLASSES, fontsize=7)
for (i, j), v in np.ndenumerate(cm):
    if v:
        axes[0].text(j, i, v, ha="center", va="center", fontsize=7)
axes[0].set_xlabel("predicted")
axes[0].set_ylabel("true")
axes[0].set_title(f"Confusion matrix: {best['model']}")
conf, hit = probs.max(1), probs.argmax(1) == y
edges = np.linspace(0, 1, 11)
idx = np.clip(np.digitize(conf, edges) - 1, 0, 9)
acc = [hit[idx == k].mean() if (idx == k).any() else np.nan for k in range(10)]
axes[1].bar(edges[:-1] + 0.05, acc, width=0.09, label="accuracy in bin")
axes[1].plot([0, 1], [0, 1], "k--", label="perfect calibration")
axes[1].set_xlabel("predicted confidence")
axes[1].set_ylabel("accuracy")
axes[1].set_title(f"Reliability diagram (ECE {best['ece']:.3f})")
axes[1].legend()
plt.tight_layout()
plt.show()

### 6.7 Learning curves (over- or under-fitting?)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for _, r in res[(res["experiment"] == main_exp) & (res["test_set"] == main_test)].iterrows():
    h = [x for x in r["history"] if "epoch" in x]
    if h:
        ax.plot(range(len(h)), [x["val_macro_f1"] for x in h], marker=".", label=r["model"])
ax.set_xlabel("epoch (stage 1 then stage 2)")
ax.set_ylabel("validation macro-F1")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## 7. Decision (fill in after running — applies the pre-registered rule from the top)
- Winner on local-test macro-F1 (E2): …
- Confidence intervals overlapping with: … → tie-break on costly-error: … → latency: …
- **Chosen model:** … because …
- What the comparison taught us (CNN vs transformer, fine-tuning vs probe, zero-shot on
  e-waste, domain gap): …

Write the chosen key into Drive so notebooks 03 and 04 use it:

In [ ]:
CHOSEN = ranked.iloc[0]["model"]  # override by hand if the rule's tie-breaks pick another
(RESULTS / "chosen_model.json").write_text(json.dumps({"model": CHOSEN, "experiment": main_exp}))
print("chosen:", CHOSEN)